In [12]:
require 'gd'
# this function show the image on the notebook
def render(file_name)
    IRuby.display "<img src='#{file_name}' />", mime: "text/html"
end

:random_font

In [59]:
# ═════════════════════════════════════════════════════════════════
#  SafeMath — expression evaluator using Ruby's Math module
#  Supports: sin, cos, tan, asin, acos, atan, sinh, cosh, tanh,
#            log (natural), ln, log10, log2, sqrt, exp, cbrt,
#            abs, floor, ceil, round, pi, e
# ═════════════════════════════════════════════════════════════════
module SafeMath
  class EvalContext
    include Math

    def pi; Math::PI; end
    def e;  Math::E;  end

    def initialize(vars)
      vars.each do |k, v|
        val = v.to_f
        define_singleton_method(k) { val }
      end
    end

    def ln(v);    Math.log(v);   end
    def log(v);   Math.log(v);   end
    def log10(v); Math.log10(v); end
    def log2(v);  Math.log2(v);  end
    def abs(v);   v.abs;         end
    def floor(v); v.floor;       end
    def ceil(v);  v.ceil;        end
    def round(v); v.round;       end
    def cbrt(v);  Math.cbrt(v);  end

    def eval_expr(expr)
      result = eval(expr)
      return nil unless result.is_a?(Numeric)
      return nil if result.respond_to?(:infinite?) && result.infinite?
      return nil if result.respond_to?(:nan?)      && result.nan?
      result.to_f
    end
  end

  def self.evaluate(expr, vars = {})
    EvalContext.new(vars).eval_expr(expr)
  rescue Math::DomainError, ZeroDivisionError, FloatDomainError
    nil
  rescue StandardError
    nil
  end
end


# ═════════════════════════════════════════════════════════════════
#  Plot — 2D function grapher built on ruby-libgd
#
#  Usage:
#    plot = Plot.new
#    plot.add("sin(x)")
#    plot.add("cos(x)", color: GD::Color.rgb(60, 130, 220))
#    plot.add("x**2 + y**2 - 25", type: :implicit)
#    plot.render("/work/graph.png")
#
#  Constructor options (all optional):
#    width:     image width  in pixels (default: 600)
#    height:    image height in pixels (default: 600)
#    xmin:      left   bound (default: -10.0)
#    xmax:      right  bound (default:  10.0)
#    ymin:      bottom bound (default: -10.0)
#    ymax:      top    bound (default:  10.0)
#    bg:        background GD::Color (default: off-white)
#    grid:      grid line  GD::Color (default: light gray)
#    axis:      axis line  GD::Color (default: medium gray)
#
#  add(expr, options) options:
#    type:       :explicit (default) or :implicit
#    color:      GD::Color (default: auto from palette)
#    steps:      sample count for explicit curves (default: 3000)
#    threshold:  zero-crossing tolerance for implicit (default: 0.30)
# ═════════════════════════════════════════════════════════════════
class Plot
  DEFAULT_PALETTE = [
    [220,  60,  60],  # red
    [ 60, 130, 220],  # blue
    [ 20, 160,  80],  # green
    [160,  60, 200],  # purple
    [220, 140,   0],  # amber
    [  0, 180, 180],  # teal
    [220, 100,  40],  # orange
    [180,  40, 100],  # pink
    [100, 100, 200],  # indigo
    [ 60, 180, 180],  # cyan
    [230, 160,  20],  # yellow
    [120, 120,  40],  # olive
  ].freeze

  def initialize(
    width:  600,
    height: 600,
    xmin:  -10.0,
    xmax:   10.0,
    ymin:  -10.0,
    ymax:   10.0,
    bg:     nil,
    grid:   nil,
    axis:   nil
  )
    @width  = width
    @height = height
    @xmin   = xmin.to_f
    @xmax   = xmax.to_f
    @ymin   = ymin.to_f
    @ymax   = ymax.to_f

    @scale_x = @width.to_f  / (@xmax - @xmin)
    @scale_y = @height.to_f / (@ymax - @ymin)

    @curves        = []
    @palette_index = 0

    # Colors stored as [r,g,b] arrays — resolved to GD::Color at render time
    @bg_rgb   = bg   || [248, 248, 245]
    @grid_rgb = grid || [220, 220, 215]
    @axis_rgb = axis || [150, 150, 145]
  end

  # ── Public API ──────────────────────────────────────────────
  #
  # Add an explicit curve:   plot.add("sin(x)")
  # Add an implicit curve:   plot.add("x**2 + y**2 - 9", type: :implicit)
  #
  def add(expr, type: :explicit, color: nil, steps: 3000, threshold: 0.30)
    resolved_color = color || next_color
    @curves << {
      expr:      expr,
      type:      type,
      color:     resolved_color,   # [r,g,b] array
      steps:     steps,
      threshold: threshold
    }
    self  # chainable: plot.add("sin(x)").add("cos(x)")
  end

  def random_font
    # system font path
    path = "/usr/share/fonts/truetype/**/*.ttf"
    Dir.glob(path).sample
  end
    
  # Render to a file and call render() for Jupyter display
  def render(path = "/work/graph.png")
    img = build_image
    img.save(path)
  end

  # Return the GD::Image object without saving (for compositing etc.)
  def to_image
    build_image
  end

  private

  # ── Coordinate helpers ──────────────────────────────────────

  def to_px(x, y)
    px = ((x - @xmin) * @scale_x).to_i
    py = (@height - (y - @ymin) * @scale_y).to_i
    [px, py]
  end

  def in_bounds?(px, py)
    px.between?(1, @width - 2) && py.between?(1, @height - 2)
  end

  # ── Color helpers ───────────────────────────────────────────

  def next_color
    rgb = DEFAULT_PALETTE[@palette_index % DEFAULT_PALETTE.size]
    @palette_index += 1
    rgb
  end

  def gd_color(rgb)
    GD::Color.rgb(rgb[0], rgb[1], rgb[2])
  end

  # ── Image construction ──────────────────────────────────────

  def build_image
    img = GD::Image.new(@width, @height)

    c_bg   = gd_color(@bg_rgb)
    c_grid = gd_color(@grid_rgb)
    c_axis = gd_color(@axis_rgb)

    # Background
    img.filled_rectangle(0, 0, @width - 1, @height - 1, c_bg)

    # Grid
    draw_grid(img, c_grid)

    # Axes
    draw_axes(img, c_axis)

    # Curves — explicit first, implicit on top
    @curves.select { |c| c[:type] == :explicit }.each { |c| draw_explicit(img, c) }
    @curves.select { |c| c[:type] == :implicit }.each { |c| draw_implicit(img, c) }

    fx = @curves.first[:expr]
    pp  @curves.first[:expr].class
    fx = @curves.first[:expr].first if @curves.first[:expr].is_a?(Array)
        
    img.text("Function: #{fx}", x: 20, y: 20, color: [0, 0, 0], size: 10, font: random_font)
    img.text("GEM: Ruby-LibGD", x: 20, y: 570, color: [0, 0, 0], size: 10, font: random_font)
      
    img
  end

  def draw_grid(img, color)
    grid_start = (@xmin.ceil + 1).to_i
    grid_end   = (@xmax.floor - 1).to_i

    (grid_start..grid_end).each do |n|
      px, _ = to_px(n,     @ymin)
      _, py = to_px(@xmin, n)
      px = px.clamp(1, @width  - 2)
      py = py.clamp(1, @height - 2)
      img.line(px, 1,          px, @height - 2, color)
      img.line(1,  py, @width - 2, py,          color)
    end
  end

  def draw_axes(img, color)
    cx, cy = to_px(0, 0)
    cx = cx.clamp(1, @width  - 2)
    cy = cy.clamp(1, @height - 2)
    img.line(1,  cy, @width - 2,  cy, color)
    img.line(cx,  1,          cx, @height - 2, color)
  end

  def draw_explicit(img, curve)
    color = gd_color(curve[:color])
    steps = curve[:steps]
    prev  = nil

    steps.times do |i|
      x = @xmin + (i.to_f / steps) * (@xmax - @xmin)
      y = SafeMath.evaluate(curve[:expr], x: x)

      unless y
        prev = nil
        next
      end

      px, py = to_px(x, y)

      unless in_bounds?(px, py)
        prev = nil
        next
      end

      # Discontinuity guard — resets across vertical asymptotes (e.g. tan)
      if prev && (py - prev[1]).abs > @height / 3
        prev = nil
      end

      img.line(prev[0], prev[1], px, py, color) if prev
      prev = [px, py]
    end
  end

  def draw_implicit(img, curve)
    color     = gd_color(curve[:color])
    threshold = curve[:threshold]

    @width.times do |px|
      @height.times do |py|
        x   = @xmin + (px.to_f / @width)  * (@xmax - @xmin)
        y   = @ymax - (py.to_f / @height) * (@ymax - @ymin)
        val = SafeMath.evaluate(curve[:expr], x: x, y: y)
        img.set_pixel(px, py, color) if val && val.abs < threshold
      end
    end
  end
end

(irb):77: warning: already initialized constant #<Class:0x00007fe60856c638>::Plot::DEFAULT_PALETTE
(irb):77: warning: previous definition of DEFAULT_PALETTE was here


:draw_implicit

In [66]:
# ═════════════════════════════════════════════════════════════════
#  Example — edit freely, the Plot class above never needs to change
# ═════════════════════════════════════════════════════════════════
plot = Plot.new(xmin: -10, xmax: 10, ymin: -10, ymax: 10)

# Explicit: y = f(x)
func = ["sin(x)", "cos(x)", ["tan(x)", steps: 6000], "log(x)", "log10(x)", "sqrt(x)", "exp(x) / 500.0", "x**2 / 10.0", "x**3 / 100.0", "asin(x)", "atan(x)", "abs(x) - 5", ["x**2 + y**2 - 5", type: :implicit, threshold: 0.35], ["x**2 + y**2 - 25", type: :implicit, threshold: 0.40], ["x**2/9.0 + y**2/4.0 - 1", type: :implicit, threshold: 0.20], ["y**2 - x**3 - x", type: :implicit, threshold: 0.25]]

# plot.add(func.sample)
# plot.add(["y**2 - x**3 - x", type: :implicit, threshold: 0.25])
plot.add("x**2 + y**2 - 9", type: :implicit)   # circle radius 3

name = "graph-#{rand}.png"
plot.render("/work/#{name}")

render(name)

String


"<img src='graph-0.4236340070401925.png' />"